In [ ]:
"""
Task-Aware CTS Prediction
=========================
Loads from pre-processed .pt files. Imports utilities from helper.py.
"""

import os, math, random
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

from helper import get_compressed_graph, relative_masking_dense

# ─────────────────────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────────────────────

PT_DIR   = "processed_graphs/"
CSV_DIR  = "dataset_with_def/"
CSV_PATH = os.path.join(CSV_DIR, "unified_manifest_normalized.csv")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

GRID_SIZE      = 8
NUM_KNOBS      = 4
HIDDEN_DIM     = 128
MAX_K          = 64
EPOCHS         = 300
LR             = 1e-3
TAU_START      = 2.0
TAU_END        = 0.5
LAMBDA_BALANCE = 0.01
LAMBDA_SKIP    = 0.05   # reduced — was fighting skew gradient early in training
DENSITY_RADII  = [0.05, 0.10, 0.20]
SKIP_DIST_PCTS = [0, 25, 50, 75, 90, 100]
SGC_HOPS       = 2

def get_k(n_ff):
    return max(16, min(MAX_K, n_ff // 8))


# ─────────────────────────────────────────────────────────────────────────────
# 1.  FEATURE EXTRACTION  (from .pt data, no re-extraction)
# ─────────────────────────────────────────────────────────────────────────────

def extract_ff_info(data):
    """
    X columns from normalize_features():
      0:x_norm  1:y_norm  2-5:dist_boundaries  6:log(area)
      7:avg_pin_cap*1000  8:total_pin_cap*1000  9:log2(drive)
      10:is_sequential  11:is_buffer  12:toggle  13:sum_toggle
      14:signal_prob  15:non_zero  16:log1p(fan_in)  17:log1p(fan_out)
    """
    X      = data['X']
    ff_idx = (X[:, 10] == 1.0).nonzero(as_tuple=True)[0]
    return ff_idx, X[ff_idx]


def sparse_csr_to_local_edges(A_csr, ff_idx_np):
    """Convert N×N sparse CSR → local FF-indexed edge list [E, 2]."""
    try:
        A_coo = A_csr.to_sparse_coo().coalesce()
        idx   = A_coo.indices().numpy()
        if idx.shape[1] == 0:
            return np.zeros((0, 2), dtype=np.int64)
    except Exception:
        return np.zeros((0, 2), dtype=np.int64)

    g2l = {int(g): l for l, g in enumerate(ff_idx_np)}
    pairs = [[g2l[int(s)], g2l[int(d)]]
             for s, d in zip(idx[0], idx[1])
             if int(s) in g2l and int(d) in g2l]
    return np.array(pairs, dtype=np.int64) if pairs else np.zeros((0, 2), dtype=np.int64)


def build_sgc_features(ff_X_raw, ff_skip_local, hops=SGC_HOPS):
    """
    SGC on FF timing subgraph: X_smooth = [X | S·X | S²·X]
    Returns [n_ff, 9*(hops+1)] = [n_ff, 27]. NOT normalized here.
    Global normalization applied after all placements loaded.
    """
    X = np.stack([
        ff_X_raw[:, 0].numpy(),   # x_norm
        ff_X_raw[:, 1].numpy(),   # y_norm
        ff_X_raw[:, 12].numpy(),  # toggle_count
        ff_X_raw[:, 13].numpy(),  # sum_toggle_count
        ff_X_raw[:, 14].numpy(),  # signal_prob
        ff_X_raw[:, 9].numpy(),   # log2(drive_strength)
        ff_X_raw[:, 7].numpy(),   # avg_pin_cap*1000
        ff_X_raw[:, 16].numpy(),  # log1p(fan_in)
        ff_X_raw[:, 17].numpy(),  # log1p(fan_out)
    ], axis=1).astype(np.float32)   # [n_ff, 9]

    n = X.shape[0]
    if len(ff_skip_local) == 0 or n < 2:
        return np.tile(X, (1, hops + 1))

    rows = np.concatenate([ff_skip_local[:, 0], ff_skip_local[:, 1], np.arange(n)])
    cols = np.concatenate([ff_skip_local[:, 1], ff_skip_local[:, 0], np.arange(n)])
    A    = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n, n))
    deg  = np.array(A.sum(axis=1)).flatten() + 1e-8
    S    = sp.diags(1.0 / np.sqrt(deg)) @ A @ sp.diags(1.0 / np.sqrt(deg))

    smoothed = [X]; cur = X.copy()
    for _ in range(hops):
        cur = S @ cur; smoothed.append(cur.copy())
    return np.concatenate(smoothed, axis=1)   # [n_ff, 27]


def spatial_grid_features(X_all, G=GRID_SIZE):
    """
    4-channel G×G grid → [4*G²] = 256.
    ch0: all-node density  — count/total, always in [0,1]
    ch1: all-node area     — min-max scaled within design so always in [0,1]
    ch2: FF-only density   — count/total, always in [0,1]
    ch3: FF toggle         — min-max scaled within design so always in [0,1]

    All channels normalized to [0,1] before accumulation so global_feats
    never has values outside a reasonable range regardless of design scale.
    """
    ch     = np.zeros((4, G, G), dtype=np.float32)
    x_z    = X_all[:, 0].numpy()
    y_z    = X_all[:, 1].numpy()
    is_seq = (X_all[:, 10] == 1.0).numpy()

    # Min-max scale area and toggle to [0,1] within this design
    # This removes the z-score scale issue while preserving relative differences
    area_raw   = X_all[:, 6].numpy()
    toggle_raw = X_all[:, 12].numpy()

    def minmax01(v):
        vmin, vmax = v.min(), v.max()
        return (v - vmin) / (vmax - vmin + 1e-8)

    area   = minmax01(area_raw)    # [0,1] relative area within this design
    toggle = minmax01(toggle_raw)  # [0,1] relative toggle within this design

    xr = x_z.max() - x_z.min(); yr = y_z.max() - y_z.min()
    x_01 = (x_z - x_z.min()) / (xr if xr > 0 else 1.0)
    y_01 = (y_z - y_z.min()) / (yr if yr > 0 else 1.0)
    gx = np.clip((x_01 * G).astype(int), 0, G-1)
    gy = np.clip((y_01 * G).astype(int), 0, G-1)

    np.add.at(ch[0], (gx, gy), 1)
    np.add.at(ch[1], (gx, gy), area)
    fm = is_seq.astype(bool)
    np.add.at(ch[2], (gx[fm], gy[fm]), 1)
    np.add.at(ch[3], (gx[fm], gy[fm]), toggle[fm])

    # Normalize each channel to sum to 1 (probability distribution over grid)
    for i in range(4):
        s = ch[i].sum()
        if s > 0: ch[i] /= s
    return ch.flatten()   # 256, all values in [0,1]


def skip_distance_features(ff_X_raw, ff_skip_local):
    """
    Percentile distribution of launch-capture distances. Fixed size: 8.
    x/y are z-scored per placement (N(0,1)), so pairwise distances
    follow a half-normal with max typically ~4-6 std units.
    We divide by sqrt(2) * expected_max ≈ 4.0 to bring into [0,1] range.
    """
    n_feat = len(SKIP_DIST_PCTS) + 2
    if len(ff_skip_local) == 0:
        return np.zeros(n_feat, dtype=np.float32)
    xy    = np.stack([ff_X_raw[:, 0].numpy(), ff_X_raw[:, 1].numpy()], axis=1)
    dists = np.sqrt(((xy[ff_skip_local[:, 0]] - xy[ff_skip_local[:, 1]])**2).sum(1))
    # Scale: max distance between two N(0,1) 2D points is typically ~4-6
    # Divide by 4.0 to bring into roughly [0,1] range
    dists = dists / 4.0
    pcts  = np.percentile(dists, SKIP_DIST_PCTS).astype(np.float32)
    return np.concatenate([pcts, [dists.mean()], [dists.std()]])


def skew_physics_features(ff_X_raw, ff_skip_local):
    """
    Physical features directly relevant to skew prediction.
    Fixed size: 7

    1. Drive strength VARIANCE within timing-path FFs:
       Skew = difference in arrival times. Asymmetric drive strength
       across paths causes asymmetric RC delay → skew.
       It's the VARIANCE that matters, not mean/min.

    2. Spatial asymmetry of launch vs capture FFs:
       If launch FFs cluster in one region and capture FFs in another,
       the CTS tool must route long unbalanced paths → high skew.
       We measure: distance between centroid(launch) and centroid(capture).

    3. Pin capacitance variance of timing-path FFs:
       High cap variance = unbalanced load on clock tree branches → skew.
    """
    n_ff = ff_X_raw.shape[0]
    zeros = np.zeros(7, dtype=np.float32)
    if len(ff_skip_local) == 0 or n_ff < 2:
        return zeros

    # Col 9: log2(drive_strength), Col 7: avg_pin_cap*1000
    drive = ff_X_raw[:, 9].numpy()    # log2(drive) — already z-scored
    cap   = ff_X_raw[:, 7].numpy()    # avg_pin_cap — already z-scored
    x_pos = ff_X_raw[:, 0].numpy()    # x_norm — z-scored
    y_pos = ff_X_raw[:, 1].numpy()    # y_norm — z-scored

    launch_idx  = ff_skip_local[:, 0]
    capture_idx = ff_skip_local[:, 1]

    # Unique FFs involved in timing paths
    timing_ffs = np.unique(np.concatenate([launch_idx, capture_idx]))

    # 1. Drive strength variance of timing-path FFs
    drive_timing = drive[timing_ffs]
    drive_var    = float(drive_timing.var()) if len(drive_timing) > 1 else 0.0
    drive_std    = float(drive_timing.std()) if len(drive_timing) > 1 else 0.0

    # 2. Spatial centroid distance: launch vs capture
    launch_cx  = x_pos[launch_idx].mean()
    launch_cy  = y_pos[launch_idx].mean()
    capture_cx = x_pos[capture_idx].mean()
    capture_cy = y_pos[capture_idx].mean()
    centroid_dist = float(np.sqrt((launch_cx-capture_cx)**2 +
                                   (launch_cy-capture_cy)**2)) / 4.0  # normalize

    # 3. Spatial spread asymmetry: std(launch_x) vs std(capture_x)
    launch_spread  = float(np.sqrt(x_pos[launch_idx].var() +
                                    y_pos[launch_idx].var()))
    capture_spread = float(np.sqrt(x_pos[capture_idx].var() +
                                    y_pos[capture_idx].var()))
    spread_ratio   = abs(launch_spread - capture_spread) /                      (launch_spread + capture_spread + 1e-8)

    # 4. Pin capacitance variance of timing-path FFs
    cap_timing = cap[timing_ffs]
    cap_var    = float(cap_timing.var()) if len(cap_timing) > 1 else 0.0
    cap_std    = float(cap_timing.std()) if len(cap_timing) > 1 else 0.0

    # 5. Fraction of FFs involved in timing paths
    frac_timing = len(timing_ffs) / max(n_ff, 1)

    return np.array([drive_var, drive_std, centroid_dist,
                     launch_spread, capture_spread, spread_ratio,
                     cap_var], dtype=np.float32)  # [7]



def ff_density_features(ff_X_raw):
    """
    FF neighbor fraction at multiple radii. Fixed size: 9.
    Returns fractions (0-1) not raw counts — design-size invariant.
    x/y are z-scored per placement so distances are in std-dev units.
    At radius 0.20 in N(0,1) space, ~16% of FFs are within range.
    Dividing by n_ff makes these comparable across designs of any size.
    """
    n = ff_X_raw.shape[0]
    if n < 2:
        return np.zeros(len(DENSITY_RADII) * 3, dtype=np.float32)
    xy = np.stack([ff_X_raw[:, 0].numpy(), ff_X_raw[:, 1].numpy()], axis=1).astype(np.float32)
    n_sample = min(n, 500)
    if n > 500:
        xy = xy[np.random.default_rng(42).choice(n, n_sample, replace=False)]
    diff  = xy[:, None, :] - xy[None, :, :]
    dists = np.sqrt((diff**2).sum(axis=-1))
    np.fill_diagonal(dists, np.inf)
    feats = []
    for r in DENSITY_RADII:
        # Normalize by (n_sample - 1) so values are fractions in [0, 1]
        c = (dists < r).sum(axis=1).astype(np.float32) / (n_sample - 1)
        feats.extend([c.mean(), c.std(), c.max()])
    return np.array(feats, dtype=np.float32)


def synthesis_flags(row):
    """6 binary flags from synthesis/placement strategy columns."""
    strat = str(row.get('synth_strategy', '')).upper()
    return np.array([
        float('AREA 0' in strat), float('AREA 2' in strat), float('DELAY' in strat),
        float(int(row.get('io_mode', 0))),
        float(int(row.get('time_driven', 0))),
        float(int(row.get('routability_driven', 0))),
    ], dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# 2.  GLOBAL NORMALIZATION
# ─────────────────────────────────────────────────────────────────────────────

def compute_global_stats(matrices):
    """
    Robust mu/std across a list of 2D numpy arrays stacked row-wise.
    Uses mean/std but clips extreme outliers first to prevent a single
    bad placement from dominating normalization statistics.
    """
    all_X = np.concatenate(matrices, axis=0)
    # Clip to 5th-95th percentile range before computing stats
    # This makes normalization robust to the outlier placements that
    # were causing mu=-71567 and sig=508461
    p5  = np.percentile(all_X, 5,  axis=0)
    p95 = np.percentile(all_X, 95, axis=0)
    all_X_clipped = np.clip(all_X, p5, p95)
    mu  = all_X_clipped.mean(axis=0).astype(np.float32)
    sig = all_X_clipped.std(axis=0).astype(np.float32)
    return mu, np.where(sig < 1e-8, 1.0, sig)

def apply_norm(X, mu, sig):
    return (X - mu) / sig


# ─────────────────────────────────────────────────────────────────────────────
# 3.  DATASET LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_dataset(pt_dir, csv_path):
    df       = pd.read_csv(csv_path)
    df_by_id = {pid: grp for pid, grp in df.groupby('placement_id')}
    pt_files = sorted([f for f in os.listdir(pt_dir) if f.endswith('.pt')])
    print(f"Found {len(pt_files)} .pt files")

    raw = []

    for pt_file in pt_files:
        pid    = pt_file.replace('.pt', '')
        if pid not in df_by_id:
            continue

        group  = df_by_id[pid]
        row0   = group.iloc[0]
        design = str(row0.get('design_name', pid.split('_')[0]))
        print(f"\n{'─'*50}\nLoading: {pid}  ({design})")

        try:
            data = torch.load(os.path.join(pt_dir, pt_file), map_location='cpu')
        except Exception as e:
            print(f"  SKIP: {e}"); continue

        X_all     = data['X']
        num_nodes = data['num_nodes']
        ff_idx, ff_X_raw = extract_ff_info(data)
        n_ff = len(ff_idx)
        if n_ff < 8:
            print(f"  SKIP: only {n_ff} FFs"); continue

        ff_idx_np = ff_idx.numpy()

        # Pre-filter edges to local FF space at load time
        # so forward() never touches the full N×N matrix
        ff_skip_local = sparse_csr_to_local_edges(data['A_skip_csr'], ff_idx_np)
        # Build weighted FF-to-FF wire proximity via 2-hop expansion.
        # A_wire²[i,j] = number of shared wire neighbors between nodes i and j.
        # When restricted to FF×FF submatrix, this gives a weighted graph where
        # high weight = many shared intermediate gates = physically close FFs.
        # relative_masking_dense then keeps only the strongest connections,
        # giving a sparse meaningful FF wire proximity graph.
        try:
            A_coo  = data['A_wire_csr'].to_sparse_coo().coalesce()
            widx   = A_coo.indices().numpy()
            wvals  = np.ones(widx.shape[1], dtype=np.float32)
            n      = data['num_nodes']
            A_sp   = sp.csr_matrix((wvals, (widx[0], widx[1])), shape=(n, n))

            # Extract FF rows, then multiply: [n_ff, N] @ [N, N] → [n_ff, N]
            # Then extract FF cols: [n_ff, n_ff] weighted 2-hop matrix
            A_ff_rows  = A_sp[ff_idx_np, :]          # [n_ff, N]
            A_ff_2hop  = (A_ff_rows @ A_sp)[:, ff_idx_np]  # [n_ff, n_ff]
            A_ff_dense = torch.tensor(A_ff_2hop.toarray(), dtype=torch.float32)

            # relative_masking_dense: keep only connections above threshold*row_max
            # This removes weak/noise connections and keeps physically meaningful ones
            from helper import relative_masking_dense
            A_ff_masked = relative_masking_dense(A_ff_dense)

            # Extract surviving edges as local FF index pairs
            r_idx, c_idx = (A_ff_masked > 0).nonzero(as_tuple=True)
            # Remove self-loops
            mask_no_self = r_idx != c_idx
            r_idx = r_idx[mask_no_self]; c_idx = c_idx[mask_no_self]
            ff_wire_local = np.stack([r_idx.numpy(), c_idx.numpy()], axis=1).astype(np.int64)                             if len(r_idx) > 0 else np.zeros((0, 2), dtype=np.int64)
        except Exception as e:
            print(f"  wire 2-hop fallback: {e}")
            ff_wire_local = np.zeros((0, 2), dtype=np.int64)

        # SGC features — raw, global norm applied after all loaded
        ff_X_smooth = build_sgc_features(ff_X_raw, ff_skip_local)   # [n_ff, 27]

        # Global fixed-size features — raw, global norm applied after
        grid_f    = spatial_grid_features(X_all)                    # 256
        skip_f    = skip_distance_features(ff_X_raw, ff_skip_local) # 8
        density_f = ff_density_features(ff_X_raw)                   # 9
        skew_f    = skew_physics_features(ff_X_raw, ff_skip_local)  # 7
        synth_f   = synthesis_flags(row0)                           # 6
        scale_f   = np.array([
            math.log(max(n_ff, 1))         / 10.0,
            math.log(max(num_nodes, 1))    / 10.0,
            math.log(max(num_nodes**2, 1)) / 20.0,
            n_ff / max(num_nodes, 1),
        ], dtype=np.float32)                                         # 4
        global_feats = np.concatenate([grid_f, skip_f, density_f,
                                       skew_f, synth_f, scale_f])   # 290

        # CTS runs
        cts_runs = []
        for _, row in group.iterrows():
            knobs = torch.tensor([
                row['z_cts_max_wire'], row['z_cts_buf_dist'],
                row['z_cts_cluster_size'], row['z_cts_cluster_dia'],
            ], dtype=torch.float32)
            targets = {
                'skew':  torch.tensor([row['z_skew_setup']],  dtype=torch.float32),
                'power': torch.tensor([row['z_power_total']], dtype=torch.float32),
                'wl':    torch.tensor([row['z_wirelength']],  dtype=torch.float32),
            }
            cts_runs.append({'knobs': knobs, 'targets': targets})

        if not cts_runs:
            print("  SKIP: no CTS runs"); continue

        # Store raw_areas for get_compressed_graph (from helper)
        raw_areas = torch.tensor(
            [[n['cell_area']] for n in data.get('nodes', [])]
            if 'nodes' in data
            else [[0.0]] * num_nodes,
            dtype=torch.float32
        )

        raw.append({
            'placement_id':  pid,
            'design_name':   design,
            'ff_X_smooth':   ff_X_smooth,       # [n_ff, 27] raw
            'global_feats':  global_feats,       # [283] raw
            'ff_skip_local': ff_skip_local,      # [E, 2]
            'ff_wire_local': ff_wire_local,      # [E, 2]
            'A_skip_csr':    data['A_skip_csr'],
            'A_wire_csr':    data['A_wire_csr'],
            'X_full':        X_all,              # [N, 18] for get_compressed_graph
            'raw_areas':     raw_areas,          # [N, 1]
            'num_nodes':     num_nodes,
            'n_ff':          n_ff,
            'k':             get_k(n_ff),
            'cts_runs':      cts_runs,
        })
        print(f"  OK  n_ff={n_ff}  k={get_k(n_ff)}  "
              f"skip={len(ff_skip_local)}  wire={len(ff_wire_local)}  "
              f"runs={len(cts_runs)}")

    if not raw:
        return []

    # ── Global normalization for BOTH feature sets ────────────────────────
    # Critical: task heads assume ~N(0,1) inputs.
    # Unnormalized global_feats with mixed scales caused 2600x loss explosion.
    print(f"\nNormalizing across {len(raw)} placements...")

    mu_sgc, sig_sgc = compute_global_stats([p['ff_X_smooth'] for p in raw])
    mu_gf,  sig_gf  = compute_global_stats(
        [p['global_feats'].reshape(1, -1) for p in raw])

    print(f"  SGC:    mu=[{mu_sgc.min():.2f},{mu_sgc.max():.2f}]  "
          f"sig=[{sig_sgc.min():.2f},{sig_sgc.max():.2f}]")
    print(f"  Global: mu=[{mu_gf.min():.2f},{mu_gf.max():.2f}]  "
          f"sig=[{sig_gf.min():.2f},{sig_gf.max():.2f}]")

    placements = []
    for p in raw:
        placements.append({
            **p,
            'ff_X_smooth':  torch.tensor(
                apply_norm(p['ff_X_smooth'], mu_sgc, sig_sgc), dtype=torch.float32),
            'global_feats': torch.tensor(
                apply_norm(p['global_feats'].reshape(1, -1), mu_gf, sig_gf).flatten(),
                dtype=torch.float32),
            'ff_skip_local':torch.tensor(p['ff_skip_local'], dtype=torch.long),
            'ff_wire_local':torch.tensor(p['ff_wire_local'], dtype=torch.long),
        })

    print(f"\n{'═'*50}")
    print(f"Loaded {len(placements)} placements")
    by_d = defaultdict(int)
    for p in placements: by_d[p['design_name']] += 1
    for d, c in sorted(by_d.items()): print(f"  {d}: {c}")

    return placements, mu_sgc, sig_sgc, mu_gf, sig_gf


# ─────────────────────────────────────────────────────────────────────────────
# 4.  MODEL
# ─────────────────────────────────────────────────────────────────────────────

class AssignmentNet(nn.Module):
    """Row-wise MLP [n_ff, 27] → C [n_ff, k]. Permutation equivariant."""
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, max_k=MAX_K):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2), nn.GELU(),
        )
        self.proto_bank = nn.Linear(hidden_dim // 2, max_k, bias=False)

    def forward(self, X, k, tau=1.0):
        logits = self.proto_bank(self.trunk(X))[:, :k]
        return F.gumbel_softmax(logits, tau=tau, hard=False, dim=-1)


def build_compressed_adj(ff_local_edges, C, n_ff, device):
    """
    Compress edge list directly to k×k without building dense n_ff×n_ff matrix.

    A_c[i,j] = sum_{(u,v) in edges} C[u,i] * C[v,j]

    This is equivalent to C.t() @ A_ff @ C but avoids allocating the
    [n_ff, n_ff] dense matrix. For 2994 FFs that saves 36MB per call.
    With p_indices giving thousands of FF-FF pairs this matters a lot.

    Relative masking applied after to keep k×k sparse.
    Gradients preserved on surviving entries.
    """
    k = C.shape[1]
    if ff_local_edges.shape[0] == 0:
        return torch.zeros(k, k, device=device)

    r = ff_local_edges[:, 0].to(device)   # [E]
    c = ff_local_edges[:, 1].to(device)   # [E]

    # C[r]: [E, k] — cluster assignments of source nodes
    # C[c]: [E, k] — cluster assignments of dest nodes
    # outer product per edge then sum: equivalent to C.t() @ A_ff @ C
    Cr = C[r]   # [E, k]
    Cc = C[c]   # [E, k]

    # A_c[i,j] = sum_e Cr[e,i] * Cc[e,j]
    A_c = Cr.t() @ Cc   # [k, k]
    # Symmetrize (undirected edges)
    A_c = A_c + A_c.t()

    return relative_masking_dense(A_c)   # sparse k×k, gradients preserved


def trace_moments(A, k):
    """
    Row-normalize A first so all entries ∈ [0,1].
    Then: [Tr(Â)/k, Tr(Â²)/k, Tr(Â³)/k, ||Â||_F/k]
    Row-norm prevents A³ explosion (raw entry 50 → 125,000 gradient bomb).
    After norm: Â³ entries ≤ 1, gradients always bounded.
    """
    A_hat = A / A.sum(dim=1, keepdim=True).clamp(min=1e-8)
    tr1   = A_hat.diagonal().sum() / k
    A2    = A_hat @ A_hat
    tr2   = A2.diagonal().sum() / k
    tr3   = (A2 @ A_hat).diagonal().sum() / k
    frob  = A_hat.norm('fro') / k
    return torch.stack([tr1, tr2, tr3, frob])


class TaskHead(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),    nn.GELU(),
            nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Linear(hidden//2, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)


class TaskAwareCTS(nn.Module):
    """
    No GNN. No P matrix. No eigendecomposition.

    encode()  : graph → phi_base [global(283)|trace(8)|z_skew(27)|z_power(27)|z_wl(54)] = [399]
    predict() : phi_base + knobs_batch [B,4] → predictions [B] per task

    Compute C once per placement, batch all 10 CTS configs through predict().
    """
    def __init__(self, sgc_dim, global_feat_dim,
                 max_k=MAX_K, hidden_dim=HIDDEN_DIM, num_knobs=NUM_KNOBS):
        super().__init__()
        self.max_k      = max_k
        self.assign_net = AssignmentNet(sgc_dim, hidden_dim, max_k)

        # Separate LayerNorms per graph type
        # Wire graph is far denser than skip graph — their trace magnitudes
        # differ by orders of magnitude. Independent norms prevent the dense
        # wire signal from squashing the sparse timing signal.
        self.trace_norm_skip = nn.LayerNorm(4)
        self.trace_norm_wire = nn.LayerNorm(4)

        head_in = global_feat_dim + 8 + 4 * sgc_dim + num_knobs

        # Shared trunk: power and WL are correlated (longer tree = more power)
        # Shared layers let them exploit mutual information per GAN-CTS paper
        self.shared_trunk = nn.Sequential(
            nn.Linear(head_in, 128), nn.GELU(),
            nn.Linear(128, 64),      nn.GELU(),
        )
        # Skew gets its own path with larger capacity + explicit interaction terms
        # The +5 is for the knob×design interaction features computed in predict()
        self.head_skew  = nn.Sequential(
            nn.Linear(head_in + 5, 128), nn.GELU(),
            nn.Linear(128, 64),          nn.GELU(),
            nn.Linear(64, 1),
        )
        # Power and WL share the trunk then branch
        self.head_power = nn.Linear(64, 1)
        self.head_wl    = nn.Linear(64, 1)

    def encode(self, ff_X, global_feats, ff_skip, ff_wire, k, tau=1.0):
        """Expensive part — run ONCE per placement."""
        device = ff_X.device
        n_ff   = ff_X.shape[0]

        C = self.assign_net(ff_X, k=k, tau=tau)   # [n_ff, k]

        # Compressed + masked adjacencies (uses relative_masking_dense from helper)
        A_skip_c = build_compressed_adj(ff_skip, C, n_ff, device)   # [k, k]
        A_wire_c = build_compressed_adj(ff_wire, C, n_ff, device)   # [k, k]

        # Trace moments on row-normalized matrices
        v_trace = torch.cat([
            self.trace_norm_skip(trace_moments(A_skip_c, k)),   # [4]
            self.trace_norm_wire(trace_moments(A_wire_c, k)),   # [4]
        ])   # [8]

        # Supernode feature pooling — task-specific
        C_norm = C / C.sum(dim=0, keepdim=True).clamp(min=1e-8)
        Z      = C_norm.t() @ ff_X                               # [k, sgc_dim]

        # Skew: max-pool over supernodes weighted by skip density.
        # Skew is a worst-case metric — the most timing-path-loaded supernode
        # dominates. Max-pooling captures this better than weighted averaging.
        skip_density = A_skip_c.sum(dim=1).clamp(min=0)          # [k]
        if skip_density.sum() > 1e-8:
            # Focus on top-k supernodes by skip density (soft max via high-temp softmax)
            attn_hard = torch.softmax(skip_density * 10.0, dim=0).unsqueeze(1)  # [k,1]
            z_skew    = (attn_hard * Z).sum(dim=0)                # [sgc_dim]
        else:
            z_skew = Z.max(dim=0)[0]                              # [sgc_dim] fallback

        # Power: toggle-activity-weighted
        # ff_X col 2 = toggle_count (globally normalized) — high toggle = high power
        toggle_w = C_norm.t() @ ff_X[:, 2].clamp(min=0).unsqueeze(1)  # [k,1]
        toggle_w = (toggle_w + 1e-8) / (toggle_w.sum() + 1e-8)
        z_power  = (toggle_w * Z).sum(dim=0)                     # [sgc_dim]

        # WL: mean + std (spatial spread across supernodes)
        z_wl = torch.cat([Z.mean(dim=0), Z.std(dim=0)])          # [2*sgc_dim]

        phi_base = torch.cat([global_feats, v_trace,
                               z_skew, z_power, z_wl])           # [283+8+d+d+2d]

        # Entropy regularization — prevents cluster collapse
        usage   = (C.sum(dim=0) / n_ff).clamp(min=1e-8)
        reg_ent = (usage * usage.log()).sum()

        # Skip concentration loss — forces C to keep launch-capture pairs
        # in the same or adjacent supernodes. This is the direct inductive
        # bias for skew: concentrated timing paths = predictable skew.
        if A_skip_c.sum() > 1e-8:
            M_norm    = (A_skip_c / A_skip_c.sum().clamp(min=1e-8)).clamp(min=1e-8)
            skip_ent  = -(M_norm * M_norm.log()).sum()
        else:
            skip_ent  = torch.zeros(1, device=device).squeeze()

        return phi_base, reg_ent, skip_ent, C

    def predict(self, phi_base, knobs_batch):
        """Cheap part — runs once per CTS config batch."""
        B   = knobs_batch.shape[0]
        phi = torch.cat([phi_base.unsqueeze(0).expand(B, -1),
                         knobs_batch], dim=1)   # [B, head_in]

        # ── Knob × Design interaction terms for skew ──────────────────────
        # Skew is highly sensitive to HOW knobs interact with design geometry.
        # With 140 training samples the head can't learn these products from
        # concatenated inputs alone — we pre-compute them explicitly.
        #
        # global_feats layout: grid(256) | skip_f(8) | density(9) | skew_phys(7) | ...
        # skip_f[5] = max skip distance (100th pct) at global_feats index 261
        # skip_f[6] = mean skip distance at index 262
        # skew_phys[2] = centroid_dist at index 256+8+9+2 = 275
        # skew_phys[5] = spread_ratio at index 278

        gf = phi_base  # [global+trace+pool]
        # Extract key design features (from normalized global_feats portion)
        max_skip  = gf[261].unsqueeze(0).expand(B)   # max timing path distance
        mean_skip = gf[262].unsqueeze(0).expand(B)   # mean timing path distance
        centroid  = gf[275].unsqueeze(0).expand(B)   # launch vs capture centroid gap
        spread_r  = gf[278].unsqueeze(0).expand(B)   # spatial spread asymmetry

        # knobs: [z_max_wire(0), z_buf_dist(1), z_cluster_size(2), z_cluster_dia(3)]
        buf_dist    = knobs_batch[:, 1]   # buffer distance knob
        cluster_dia = knobs_batch[:, 3]   # cluster diameter knob
        cluster_sz  = knobs_batch[:, 2]   # cluster size knob
        max_wire    = knobs_batch[:, 0]   # max wire knob

        # Physical interaction features:
        # buf_dist × max_skip: small buf_dist with long paths = more buffer stages = skew risk
        # cluster_dia × centroid: cluster too small for timing path spread = imbalance
        # cluster_dia × spread_ratio: asymmetric design needs larger cluster to balance
        # max_wire × mean_skip: wire budget relative to average path length
        interactions = torch.stack([
            buf_dist    * max_skip,      # buffer distance vs worst-case path
            cluster_dia * centroid,      # cluster scope vs timing path spread
            cluster_dia * spread_r,  # cluster scope vs spatial asymmetry
            max_wire    * mean_skip,     # wire budget vs average path
            cluster_sz  * max_skip,      # cluster size vs path length
        ], dim=1)   # [B, 5]

        # Skew head gets full phi + explicit interactions
        phi_skew = torch.cat([phi, interactions], dim=1)   # [B, head_in + 5]
        y_skew   = self.head_skew(phi_skew).squeeze(-1)

        # Power + WL: shared trunk, no interaction terms needed
        shared  = self.shared_trunk(phi)
        y_power = self.head_power(shared).squeeze(-1)
        y_wl    = self.head_wl(shared).squeeze(-1)

        return {'skew': y_skew, 'power': y_power, 'wl': y_wl}


# ─────────────────────────────────────────────────────────────────────────────
# 5.  TRAINING + EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def get_tau(epoch, anneal=200):
    p = min(epoch / anneal, 1.0)
    return TAU_END + 0.5 * (TAU_START - TAU_END) * (1 + math.cos(math.pi * p))


def run_placement(model, pl, device, tau, task_weights=None):
    ff_X   = pl['ff_X_smooth'].to(device)
    g_feat = pl['global_feats'].to(device)
    skip   = pl['ff_skip_local'].to(device)
    wire   = pl['ff_wire_local'].to(device)
    k      = pl['k']

    # Encode once
    phi_base, reg_ent, skip_ent, _ = model.encode(ff_X, g_feat, skip, wire, k, tau)

    # Batch all CTS knobs
    knobs_batch = torch.stack([c['knobs'].to(device) for c in pl['cts_runs']])
    t_skew  = torch.stack([c['targets']['skew'].to(device).squeeze()  for c in pl['cts_runs']])
    t_power = torch.stack([c['targets']['power'].to(device).squeeze() for c in pl['cts_runs']])
    t_wl    = torch.stack([c['targets']['wl'].to(device).squeeze()    for c in pl['cts_runs']])

    preds = model.predict(phi_base, knobs_batch)

    if task_weights is None:
        task_weights = {'skew': 2.0, 'power': 1.0, 'wl': 1.0}

    l_skew  = F.l1_loss(preds['skew'],  t_skew)
    l_power = F.l1_loss(preds['power'], t_power)
    l_wl    = F.l1_loss(preds['wl'],    t_wl)

    task_loss = (task_weights['skew']  * l_skew
               + task_weights['power'] * l_power
               + task_weights['wl']    * l_wl)

    total = task_loss + LAMBDA_BALANCE * reg_ent + LAMBDA_SKIP * skip_ent

    per_task = {'skew': l_skew.item(), 'power': l_power.item(), 'wl': l_wl.item()}

    return total, \
           {t: preds[t].detach().cpu().tolist() for t in ['skew','power','wl']}, \
           {'skew': t_skew.cpu().tolist(),
            'power':t_power.cpu().tolist(),
            'wl':   t_wl.cpu().tolist()}, \
           per_task


def evaluate(model, placements, device):
    model.eval()
    all_p = defaultdict(list); all_t = defaultdict(list)
    with torch.no_grad():
        for pl in placements:
            _, preds, targets, _ = run_placement(model, pl, device, tau=TAU_END)
            for task in ['skew','power','wl']:
                all_p[task].extend(preds[task])
                all_t[task].extend(targets[task])
    return {task: float(np.abs(np.array(all_p[task]) -
                               np.array(all_t[task])).mean())
            for task in ['skew','power','wl']}


def train_and_evaluate(placements, sgc_dim, global_feat_dim, device):
    print(f"\n{'═'*65}")
    print(f"Leave-one-placement-out CV  |  {len(placements)} placements")
    print(f"SGC={sgc_dim}  Global={global_feat_dim}  MAX_K={MAX_K}  "
          f"EPOCHS={EPOCHS}  LR={LR}")
    print(f"{'═'*65}")

    all_results = defaultdict(lambda: defaultdict(list))

    for hold_idx, held in enumerate(placements):
        train_set = [p for i,p in enumerate(placements) if i != hold_idx]
        design    = held['design_name']

        print(f"\n[{hold_idx+1}/{len(placements)}]  held={held['placement_id']}"
              f"  design={design}  n_ff={held['n_ff']}  k={held['k']}"
              f"  train_n={len(train_set)}")

        model = TaskAwareCTS(
            sgc_dim=sgc_dim, global_feat_dim=global_feat_dim,
            max_k=MAX_K, hidden_dim=HIDDEN_DIM, num_knobs=NUM_KNOBS,
        ).to(device)
        print(f"  params={sum(p.numel() for p in model.parameters()):,}")

        optimizer  = Adam(model.parameters(), lr=LR)
        best_val   = float('inf')
        best_state = None

        # GradNorm: dynamic task weights, updated every epoch
        task_weights   = {'skew': 2.0, 'power': 1.0, 'wl': 1.0}
        loss_ema       = {'skew': None, 'power': None, 'wl': None}
        GRADNORM_ALPHA = 0.12

        for epoch in range(EPOCHS):
            model.train()
            tau = get_tau(epoch)
            random.shuffle(train_set)
            epoch_loss = 0.

            for pl in train_set:
                optimizer.zero_grad()
                total, _, _, per_task = run_placement(
                    model, pl, device, tau, task_weights)
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_loss += total.item()

            # GradNorm: update weights based on relative loss rates
            for t in ['skew','power','wl']:
                l = per_task[t]
                if loss_ema[t] is None:
                    loss_ema[t] = l
                else:
                    loss_ema[t] = 0.9 * loss_ema[t] + 0.1 * l
            # Direct-loss weighting: higher loss task gets more gradient
            # (was inverse — that was upweighting the EASIEST task, wrong)
            total_l = sum(loss_ema[t] + 1e-8 for t in ['skew','power','wl'])
            for t in ['skew','power','wl']:
                target_w = (loss_ema[t] + 1e-8) / total_l * 3.0
                task_weights[t] = (1 - GRADNORM_ALPHA) * task_weights[t] +                                    GRADNORM_ALPHA * target_w

            if epoch % 50 == 0 or epoch == EPOCHS - 1:
                val  = evaluate(model, [held], device)
                avg  = sum(val.values()) / 3
                flag = ''
                if avg < best_val:
                    best_val   = avg
                    best_state = {k: v.cpu().clone()
                                  for k, v in model.state_dict().items()}
                    flag = '  ← best'
                print(f"  ep{epoch:3d} | train={epoch_loss/len(train_set):.4f} | "
                      f"skew={val['skew']:.4f}  power={val['power']:.4f}  "
                      f"wl={val['wl']:.4f}{flag}")

        if best_state:
            model.load_state_dict({k: v.to(device) for k,v in best_state.items()})
        final = evaluate(model, [held], device)
        print(f"\n  ── FINAL  {held['placement_id']} ──")
        for task in ['skew','power','wl']:
            print(f"     {task:6s}: {final[task]:.4f}")
        for task, mae in final.items():
            all_results[design][task].append(mae)

    # Summary
    print(f"\n{'═'*65}\nRESULTS (z-score MAE)\n{'═'*65}")
    all_maes = defaultdict(list)
    for design, tm in sorted(all_results.items()):
        print(f"\n  {design}:")
        for task in ['skew','power','wl']:
            vals = tm[task]
            print(f"    {task:6s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}  (n={len(vals)})")
            all_maes[task].extend(vals)
    print(f"\n  OVERALL:")
    for task in ['skew','power','wl']:
        print(f"    {task:6s}: {np.mean(all_maes[task]):.4f}")
    print(f"{'═'*65}\n")
    return all_results


# ─────────────────────────────────────────────────────────────────────────────
# 6.  MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\nPT:     {PT_DIR}\nCSV:    {CSV_PATH}\n")

    result = load_dataset(PT_DIR, CSV_PATH)
    if not result:
        print("ERROR: No placements loaded."); exit(1)

    placements, mu_sgc, sig_sgc, mu_gf, sig_gf = result

    sgc_dim         = placements[0]['ff_X_smooth'].shape[1]    # 27
    global_feat_dim = placements[0]['global_feats'].shape[0]   # 283

    print(f"\nDimensions:")
    print(f"  SGC per-FF : {sgc_dim}")
    print(f"  Global feat: {global_feat_dim}")
    print(f"  Head input : {global_feat_dim + 8 + 4*sgc_dim + NUM_KNOBS}")

    train_and_evaluate(placements, sgc_dim, global_feat_dim, device)

Device: cuda
PT:     processed_graphs/
CSV:    dataset_with_def/unified_manifest_normalized.csv

Found 140 .pt files

──────────────────────────────────────────────────
Loading: aes_run_20260306_173215  (aes)
  OK  n_ff=2994  k=64  skip=5595  wire=434938  runs=10

──────────────────────────────────────────────────
Loading: aes_run_20260306_175900  (aes)
  OK  n_ff=2994  k=64  skip=5596  wire=382858  runs=10

──────────────────────────────────────────────────
Loading: aes_run_20260306_180433  (aes)
  OK  n_ff=2994  k=64  skip=5596  wire=432143  runs=10

──────────────────────────────────────────────────
Loading: aes_run_20260306_181625  (aes)
  OK  n_ff=2994  k=64  skip=5595  wire=398523  runs=10

──────────────────────────────────────────────────
Loading: aes_run_20260306_183245  (aes)
  OK  n_ff=2994  k=64  skip=5596  wire=382087  runs=10

──────────────────────────────────────────────────
Loading: aes_run_20260306_190630  (aes)
  OK  n_ff=2994  k=64  skip=5596  wire=381955  runs=10



KeyboardInterrupt: 